<a href="https://colab.research.google.com/github/KekeliIsHere/galamsey-detection-ml/blob/main/galamsey_detector_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!git clone https://github.com/KekeliIsHere/galamsey-detection-ml.git

Cloning into 'galamsey-detection-ml'...
remote: Enumerating objects: 218, done.
remote: Counting objects: 100% (218/218), done.
remote: Compressing objects: 100% (214/214), done.
remote: Total 218 (delta 0), reused 214 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (218/218), 13.15 MiB | 749.00 KiB/s, done.


In [2]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

In [6]:
dataset = tf.keras.preprocessing.image_dataset_from_directory(
    "galamsey-detection-ml/dataset",
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=(224,224),
    batch_size=8
)

validation_dataset = tf.keras.preprocessing.image_dataset_from_directory(
    "galamsey-detection-ml/dataset",
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=(224,224),
    batch_size=8
)

Found 100 files belonging to 2 classes.
Using 80 files for training.
Found 100 files belonging to 2 classes.
Using 20 files for validation.


In [7]:
#Normalizaation. Helps the NN learn better
normalization_layer = layers.Rescaling(1./255)

dataset = dataset.map(lambda x, y: (normalization_layer(x), y))
validation_dataset = validation_dataset.map(lambda x, y: (normalization_layer(x), y))

In [8]:
#Loading Up Pre-Trained Model
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(224,224,3),
    include_top=False,
    weights="imagenet"
)

base_model.trainable = False

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [10]:
#Classifier
model = keras.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(1, activation='sigmoid')
])

In [11]:
#Compiling
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [12]:
#Training the Model
history = model.fit(
    dataset,
    validation_data=validation_dataset,
    epochs=10
)

Epoch 1/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 33s 2s/step - accuracy: 0.3250 - loss: 0.9408 - val_accuracy: 0.5500 - val_loss: 0.6679
Epoch 2/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step - accuracy: 0.5750 - loss: 0.6741 - val_accuracy: 0.7000 - val_loss: 0.5420
Epoch 3/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 42ms/step - accuracy: 0.7500 - loss: 0.5099 - val_accuracy: 0.8500 - val_loss: 0.4238
Epoch 4/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step - accuracy: 0.8500 - loss: 0.4243 - val_accuracy: 0.8500 - val_loss: 0.3625
Epoch 5/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 37ms/step - accuracy: 0.9000 - loss: 0.3519 - val_accuracy: 0.9000 - val_loss: 0.3179
Epoch 6/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step - accuracy: 0.9250 - loss: 0.3224 - val_accuracy: 0.9000 - val_loss: 0.2959
Epoch 7/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step - accuracy: 0.9250 - loss: 0.2772 - val_accuracy: 0.8000 - val_loss: 0.3127
Epoch 8/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - accuracy: 0.9125 - loss: 0.2570 - val_accuracy: 0.9000 - va

In [ ]:
model.save("galamsey_detector_model.h5")